# AutoDock Tutorial
### Notebook 03. Ligand Preparation for Docking  

**Version 1.0.0 - April, 2026. Monterrey**


**Authors:** 
[Ana C. Murrieta ](https://orcid.org/0000-0002-7619-8880) and [Flavio F. Contreras-Torres](https://orcid.org/0000-0003-2375-131X). Tecnológico de Monterrey.



[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NanoBiostructuresRG/AutodockTutorial/blob/main/notebooks/pdb_structure_preparation.ipynb)

---


## Contents


This notebook begins with the definition of the computational environment required for sequence hand


The activities are structured to be completed in approximately **60 to 120 minutes**. However, this is your personal workspace—we encourage you to add your own code cells, inspect additional PDB entries, and test alternative structural decisions as you build confidence with the workflow. The more you explore, the better prepared you'll be for the receptor preparation and docking steps that follow.


---

## How to work
1. **Open the notebook**: Click on the "Open in Colab" button or use the link provided to open the tutorial in **Google Colab**.
2. **Create your workspace**: Once open, go to **File > Save a copy in Drive**. This is vital! You must work on your own copy to ensure your progress is saved.
    - **Tip**: Look at the top-left corner. If you see "Copy of...", you are ready to go!
3. **Solve the exercises**: Complete the missing parts of the code where indicated. You can run each cell to test your progress (Shift + Enter).
4. When you finish:
    - **Rename** your file following the convention:
        - `YourID_preparation_docking.ipynb`
    - **Download the file**: Go to File > Download > Download .ipynb.
    - **Upload** the downloaded file to the **CANVAS assignment module**.

**Do not** modify the notebook structure or function signatures unless explicitly stated.


---

### Theoretical Background

## Open Babel 

[Open Babel](https://openbabel.org/) is a widely used open-source chemical toolbox that provides a comprehensive suite of tools for converting, manipulating, and analyzing chemical data. It supports over 100 chemical file formats, making it a versatile choice for tasks like ligand preparation for docking. Open Babel excels at rapid format conversion and basic structure manipulation, but it may not always capture complex chemical features with the same level of accuracy as more specialized tools. For example, while it can convert a ligand from SDF to PDBQT format, it may not always preserve detailed stereochemistry or handle macrocycles as effectively as other software. However, its speed and broad format support make it a popular choice for high-throughput virtual screening workflows where rapid processing is essential.



## Meeko: Molecule Parametrization and Software Interoperability

[Meeko](https://github.com/forlilab/Meeko) is a Python package developed for the parametrization of ligands and receptors within the AutoDock suite. It utilizes the **RDKit** library for chemical perception, which allows it to define molecular properties such as connectivity, bond orders, and formal charges independently of atom-naming conventions.

### Technical Characteristics:  
* **Scope of Functionality:** Meeko is a **parametrization tool** and does not include functions for **energy minimization** or geometric optimization. Molecules must be prepared with 3D coordinates prior to processing.
* **Protonation Handling:** The software perceives and preserves the **protonation state** existing in the input file. It does not dynamically calculate $pK_a$ values to add or remove protons; instead, it relies on explicit hydrogens provided by the user or external tools to match its residue templates.
* **Specialized Methods:** It supports advanced docking protocols, such as **macrocycle flexibility**, **explicitly hydrated docking**, and **reactive docking**.
* **Computational Performance:** In benchmarks involving 10,000 molecules, Meeko completed processing in 37.3 seconds, compared to 8.4 seconds for Open Babel and 2,200 seconds for MGLTools.
* **File Formats and Interoperability:** The package facilitates the use of the **SDF** format for docking inputs and outputs. Additionally, it allows for the conversion of docking results back into RDKit molecules, enabling integration with Molecular Dynamics (MD) and other computational chemistry pipelines.



### Step 0. Computational Environment and Required Tools

Before starting receptor and ligand preparation, verify that the working environment includes the software required for structure inspection, protonation, visualization, ligand refinement, and docking file generation.

This notebook relies on the following **Python libraries**:
- **Biopython** – for PDB/PQR structure parsing and manipulation (Steps 1–3, 5A)
- **RDKit** – for ligand validation and inspection (Step 4A)
- **pathlib**, **shutil**, **subprocess** – standard library modules for file handling and external command execution
- **PyYAML** – for loading atom-type rules in Step 5A

The following **external command-line tools** are required and must be available in the active environment:

| Tool | Purpose | Step(s) | Verification Command |
|------|---------|---------|---------------------|
| **PDB2PQR** | Receptor protonation at pH 7.4 | Step 3 | `pdb2pqr30` or `pdb2pqr` |
| **Open Babel** | Ligand hydrogenation and minimization | Steps 4A, 4B, 5B | `obabel` and `obminimize` |
| **Meeko** or **ADFR** | Receptor PDBQT generation | Step 5A | `mk_prepare_receptor.py` or `prepare_receptor` |
| **PyMOL** (optional) | Visual inspection of structures | All steps | `pymol` |

Additionally, Step 5A requires an **atom rules configuration file** (`atom_rules.yaml`), which should be included in the tutorial repository.

This file defines protein atom names and two-letter element mappings used during PQR → PDB conversion.

> **Note:** This notebook assumes that all required tools are already installed and accessible from the active environment. If any tool is missing, the corresponding step will fail with a descriptive error message.

You can verify the availability of each tool by running the following checks:

```python
import shutil

# PDB2PQR
print("PDB2PQR:", shutil.which("pdb2pqr30") or shutil.which("pdb2pqr"))

# Open Babel
print("obabel:", shutil.which("obabel") or shutil.which("babel"))
print("obminimize:", shutil.which("obminimize"))

# Meeko / ADFR
print("Meeko/ADFR:", 
      shutil.which("mk_prepare_receptor.py") or 
      shutil.which("mk_prepare_receptor") or 
      shutil.which("prepare_receptor"))

# Optional: PyMOL
print("PyMOL:", shutil.which("pymol"))
```

If any result is `None`, the corresponding tool is not available in the active environment and should be installed before proceeding.

## Step 1. Select the Final Working Receptor

The receptor selected in Notebook 3 will be used as the working structure for docking preparation. In this notebook, the reconstructed and UniProt-renumbered model `PPARG_P37231_231_505.B99990002_renum_uniprot.pdb` will serve as the reference receptor for structural inspection, chemical preparation, ligand-based box definition, and final conversion to docking-compatible format.

In [3]:
# Step 1. Select the final working receptor

from pathlib import Path
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa

# -------------------------------------------------------------------
# Auto-detect project root
# -------------------------------------------------------------------
notebook_dir = Path.cwd().resolve()
project_root = None

for parent in [notebook_dir] + list(notebook_dir.parents):
    if (parent / "docking" / "homology").exists():
        project_root = parent
        break

if project_root is None:
    raise FileNotFoundError(
        "\nCould not locate project root.\n"
        "   Make sure you're running this notebook from within the AutodockTutorial folder."
    )

# -------------------------------------------------------------------
# Find the renumbered receptor
# -------------------------------------------------------------------
homology_dir = project_root / "docking" / "homology"
candidates = list(homology_dir.glob("*_renum_uniprot.pdb"))

if not candidates:
    raise FileNotFoundError(
        f"\nNo renumbered receptor found in:\n   {homology_dir}\n\n"
        "   Did you complete Notebook 2, Step 8?\n"
        "   Expected file pattern: *_renum_uniprot.pdb"
    )

# Use the most recent one
receptor_path = max(candidates, key=lambda p: p.stat().st_mtime)

# -------------------------------------------------------------------
# Validate the file
# -------------------------------------------------------------------
parser = PDBParser(QUIET=True)

try:
    structure = parser.get_structure("receptor", receptor_path)
    residue_count = sum(1 for r in structure.get_residues() if is_aa(r, standard=True))
    chain_ids = sorted({c.id for c in structure.get_chains()})
except Exception as e:
    raise ValueError(
        f"\nCould not parse receptor file:\n   {receptor_path}\n\n"
        f"   Error: {e}"
    )

print("Working receptor selected successfully:")
print(f"   File: {receptor_path.name}")
print(f"   Chain(s): {', '.join(chain_ids)}")
print(f"   Protein residues: {residue_count}")
print(f"   Path: {receptor_path.resolve()}")

Working receptor selected successfully:
   File: PPARG_P37231_231_505.B99990002_renum_uniprot.pdb
   Chain(s): A
   Protein residues: 275
   Path: C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\homology\PPARG_P37231_231_505.B99990002_renum_uniprot.pdb


> **Output note:**  
> In this step, the final receptor file to be used in downstream docking preparation is identified and validated. This confirms that the renumbered reconstructed model generated at the end of Notebook 2 is present, readable, and structurally interpretable as a protein receptor file.  
>
> In this example, the selected working receptor was `PPARG_P37231_231_505.B99990002_renum_uniprot.pdb`. The file was successfully parsed, and the structure was confirmed to contain a single receptor chain (**A**) with **275 protein residues**, consistent with the reconstructed model selected and finalized previously.  
>
> In practical terms, this step establishes the exact receptor structure that will be carried forward into the preparation workflow and ensures that the notebook proceeds from a valid, already-renumbered protein model rather than from an earlier intermediate file.

## Step 2. Inspect and clean the receptor structure

After selecting the final working receptor, the next step is to inspect its structural content and generate a cleaned protein-only file for downstream docking preparation.

At this stage, the receptor is examined to confirm that the expected chain is present and that the structure does not contain unwanted components such as water molecules, ligands, ions, or other non-protein residues that could interfere with receptor preparation. The inspection also checks for basic structural issues, including residues with insertion codes, protein residues lacking Cα atoms, and atoms with alternate locations.

The cleaning procedure is intentionally conservative. Only the selected receptor chain is retained, and only standard amino-acid residues are written to the cleaned output file. Water molecules, hetero atoms, and other chains are excluded. Alternate locations are reported at this stage, but they are not automatically resolved here.

In practical terms, this step provides a validated and simplified receptor structure that is easier to inspect visually and more suitable for the chemical preparation steps that follow.

In [4]:
# Step 2. Inspect and clean the receptor structure

from pathlib import Path
from Bio.PDB import PDBParser, PDBIO, Select
from Bio.PDB.Polypeptide import is_aa

# -------------------------------------------------------------------
# Input from Step 1
# -------------------------------------------------------------------
if "receptor_path" not in globals():
    raise NameError(
        "Variable 'receptor_path' was not found. Please run Step 1 first."
    )

parser = PDBParser(QUIET=True)
structure = parser.get_structure("receptor", str(receptor_path))
model = structure[0]

# -------------------------------------------------------------------
# Detect and validate chains
# -------------------------------------------------------------------
chain_ids = sorted({chain.id for chain in model})

if not chain_ids:
    raise ValueError("No chains were found in the receptor structure.")

if len(chain_ids) == 1:
    target_chain_id = chain_ids[0]
else:
    raise ValueError(
        "Multiple chains were found in the receptor structure: "
        f"{', '.join(chain_ids)}\n"
        "Please define the intended receptor chain explicitly before cleaning."
    )

target_chain = model[target_chain_id]

# -------------------------------------------------------------------
# Inspect the selected chain
# -------------------------------------------------------------------
protein_residues = []
nonprotein_residues = []
waters = []
insertion_code_residues = []
residues_missing_ca = []
altloc_atoms = []

water_names = {"HOH", "WAT", "H2O"}

for residue in target_chain:
    hetflag, resseq, icode = residue.id
    resname = residue.get_resname().strip()

    if resname in water_names or hetflag == "W":
        waters.append(residue)
        continue

    if is_aa(residue, standard=True):
        protein_residues.append(residue)

        if icode.strip():
            insertion_code_residues.append((residue.resname, resseq, icode.strip()))

        if "CA" not in residue:
            residues_missing_ca.append((residue.resname, resseq, icode.strip() or "-"))

        for atom in residue:
            if atom.is_disordered() and atom.get_altloc().strip():
                altloc_atoms.append(
                    (residue.resname, resseq, icode.strip() or "-", atom.get_name(), atom.get_altloc().strip())
                )
    else:
        nonprotein_residues.append(residue)

other_chains = [cid for cid in chain_ids if cid != target_chain_id]

# -------------------------------------------------------------------
# Cleaning policy
# Keep only:
# - selected chain
# - standard amino-acid residues
# Remove:
# - waters
# - hetero atoms / ligands / ions
# - other chains
#
# Note:
# Alternate locations are only reported here, not resolved automatically.
# -------------------------------------------------------------------
class CleanProteinSelect(Select):
    def __init__(self, keep_chain_id):
        self.keep_chain_id = keep_chain_id

    def accept_chain(self, chain):
        return chain.id == self.keep_chain_id

    def accept_residue(self, residue):
        return is_aa(residue, standard=True)

# -------------------------------------------------------------------
# Save cleaned receptor
# -------------------------------------------------------------------
cleaned_path = receptor_path.with_name(receptor_path.stem + "_cleaned.pdb")

io = PDBIO()
io.set_structure(structure)
io.save(str(cleaned_path), select=CleanProteinSelect(target_chain_id))

# Re-parse cleaned structure for validation
cleaned_structure = parser.get_structure("cleaned_receptor", str(cleaned_path))
cleaned_model = cleaned_structure[0]
cleaned_chain_ids = sorted({chain.id for chain in cleaned_model})
cleaned_residue_count = sum(
    1 for residue in cleaned_structure.get_residues() if is_aa(residue, standard=True)
)

# -------------------------------------------------------------------
# Report
# -------------------------------------------------------------------
print("Receptor inspection summary")
print("-" * 60)
print(f"Input file:            {receptor_path.name}")
print(f"Selected chain:        {target_chain_id}")
print(f"All chain(s) found:    {', '.join(chain_ids)}")
print(f"Other chain(s):        {', '.join(other_chains) if other_chains else 'None'}")
print(f"Protein residues:      {len(protein_residues)}")
print(f"Non-protein residues:  {len(nonprotein_residues)}")
print(f"Waters:                {len(waters)}")
print(f"Residues with icode:   {len(insertion_code_residues)}")
print(f"Residues missing CA:   {len(residues_missing_ca)}")
print(f"Atoms with altloc:     {len(altloc_atoms)}")

if insertion_code_residues:
    print("\nResidues with insertion codes:")
    for resname, resseq, icode in insertion_code_residues[:10]:
        print(f"  - {resname} {resseq} (icode: {icode})")
    if len(insertion_code_residues) > 10:
        print("  ...")

if residues_missing_ca:
    print("\nProtein residues missing CA atoms:")
    for resname, resseq, icode in residues_missing_ca[:10]:
        print(f"  - {resname} {resseq} (icode: {icode})")
    if len(residues_missing_ca) > 10:
        print("  ...")

if altloc_atoms:
    print("\nAtoms with alternate locations (reported, not resolved automatically):")
    for resname, resseq, icode, atom_name, altloc in altloc_atoms[:10]:
        print(f"  - {resname} {resseq} (icode: {icode}) atom {atom_name} altloc {altloc}")
    if len(altloc_atoms) > 10:
        print("  ...")

print("\nCleaned receptor saved successfully:")
print(f"  File: {cleaned_path.name}")
print(f"  Path: {cleaned_path.resolve()}")
print(f"  Chain(s) retained: {', '.join(cleaned_chain_ids)}")
print(f"  Protein residues retained: {cleaned_residue_count}")

# Optional: update working variable for downstream steps
receptor_cleaned_path = cleaned_path

Receptor inspection summary
------------------------------------------------------------
Input file:            PPARG_P37231_231_505.B99990002_renum_uniprot.pdb
Selected chain:        A
All chain(s) found:    A
Other chain(s):        None
Protein residues:      275
Non-protein residues:  0
Waters:                0
Residues with icode:   0
Residues missing CA:   0
Atoms with altloc:     0

Cleaned receptor saved successfully:
  File: PPARG_P37231_231_505.B99990002_renum_uniprot_cleaned.pdb
  Path: C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\homology\PPARG_P37231_231_505.B99990002_renum_uniprot_cleaned.pdb
  Chain(s) retained: A
  Protein residues retained: 275


> **Output note:**  
> In this step, the selected receptor structure is inspected and cleaned to confirm that it is already suitable for downstream preparation. This validation checks whether the file contains only the expected protein chain and whether any unwanted structural components or basic anomalies remain after the reconstruction and renumbering steps.  
>
> In this example, the receptor file `PPARG_P37231_231_505.B99990002_renum_uniprot.pdb` was confirmed to contain a single chain (**A**) with **275 protein residues**. No non-protein residues, water molecules, insertion codes, missing Cα atoms, or alternate atom locations were detected. This indicates that the receptor is structurally clean and does not require additional manual cleanup at this stage.  
>
> A cleaned receptor file was then generated as `PPARG_P37231_231_505.B99990002_renum_uniprot_cleaned.pdb`, retaining the same single chain (**A**) and the full set of **275 protein residues**.  
>
> In practical terms, this step confirms that the final receptor model is already well formatted, free of obvious structural contaminants or inconsistencies, and ready for chemical preparation in the next stage of the docking workflow.

## Step 3. Define receptor chemical state

After structural inspection and cleanup, the receptor must be prepared in a chemically meaningful form for docking. At this stage, hydrogen atoms are added and the protonation state of ionizable groups is assigned under physiologically reasonable conditions.

In this notebook, receptor protonation is carried out with **PDB2PQR**, using a standard protein force field and a target pH close to physiological conditions. This step helps generate a receptor model in which titratable residues are treated consistently and polar interactions can be represented more realistically during downstream docking preparation.

The main output of this step is a **protonated receptor in PQR format**, which includes updated atomic coordinates together with charge and radius information assigned during preparation. This file provides a chemically defined receptor state that can be used as the basis for subsequent ligand compatibility checks and docking-file generation.

In [7]:
# Step 3. Define receptor chemical state
# Add hydrogens and assign protonation with PDB2PQR under near-physiological conditions.

from pathlib import Path
import shutil
import subprocess

# -------------------------------------------------------------------
# Resolve input receptor from Step 2 or from disk
# -------------------------------------------------------------------
if "receptor_cleaned_path" in globals():
    input_receptor = Path(receptor_cleaned_path)
else:
    notebook_dir = Path.cwd().resolve()
    project_root = None

    for parent in [notebook_dir] + list(notebook_dir.parents):
        if (parent / "docking" / "homology").exists():
            project_root = parent
            break

    if project_root is None:
        raise FileNotFoundError(
            "\nCould not locate project root.\n"
            "Make sure you're running this notebook from within the AutodockTutorial folder."
        )

    homology_dir = project_root / "docking" / "homology"
    candidates = list(homology_dir.glob("*_cleaned.pdb"))

    if not candidates:
        raise FileNotFoundError(
            "\nNo cleaned receptor file was found.\n"
            "Please run Step 2 first, or make sure a file matching '*_cleaned.pdb' exists in:\n"
            f"{homology_dir}"
        )

    input_receptor = max(candidates, key=lambda p: p.stat().st_mtime)

if not input_receptor.exists():
    raise FileNotFoundError(f"Cleaned receptor file not found: {input_receptor}")

print("Using cleaned receptor:")
print(f"  File: {input_receptor.name}")
print(f"  Path: {input_receptor.resolve()}")

# -------------------------------------------------------------------
# Locate PDB2PQR executable
# -------------------------------------------------------------------
pdb2pqr_exe = shutil.which("pdb2pqr30") or shutil.which("pdb2pqr")

if pdb2pqr_exe is None:
    raise EnvironmentError(
        "\nPDB2PQR was not found in the active environment.\n"
        "Please install PDB2PQR and make sure it is available from the command line."
    )

# -------------------------------------------------------------------
# Output file
# -------------------------------------------------------------------
output_dir = input_receptor.parent
receptor_pqr_path = output_dir / input_receptor.name.replace(".pdb", "_pH74.pqr")

# -------------------------------------------------------------------
# PDB2PQR settings for version 3.6.2
# -------------------------------------------------------------------
forcefield = "AMBER"
target_ph = 7.4

cmd = [
    pdb2pqr_exe,
    "--ff", forcefield,
    "--with-ph", str(target_ph),
    str(input_receptor),
    str(receptor_pqr_path),
]

print("\nRunning PDB2PQR...")
print("Command:")
print(" ".join(cmd))

# -------------------------------------------------------------------
# Run PDB2PQR with timeout protection
# -------------------------------------------------------------------
try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=300
    )
except subprocess.TimeoutExpired:
    raise TimeoutError(
        "\nPDB2PQR timed out after 300 seconds.\n"
        "Check the input receptor or environment configuration."
    )

if result.returncode != 0:
    raise RuntimeError(
        "\nPDB2PQR failed.\n\n"
        f"STDOUT:\n{result.stdout}\n\n"
        f"STDERR:\n{result.stderr}"
    )

# -------------------------------------------------------------------
# Basic validation
# -------------------------------------------------------------------
if not receptor_pqr_path.exists():
    raise FileNotFoundError(f"PQR output was not created: {receptor_pqr_path}")

atom_count = 0
hydrogen_like_count = 0

with open(receptor_pqr_path, "r", encoding="utf-8") as fh:
    for line in fh:
        if line.startswith(("ATOM", "HETATM")):
            atom_count += 1
            atom_name = line[12:16].strip()
            if atom_name.startswith("H"):
                hydrogen_like_count += 1

if atom_count == 0:
    raise ValueError("The PQR output contains no atom records.")

if hydrogen_like_count == 0:
    print("Warning: no hydrogen atom names were detected explicitly in the PQR file.")
    print("Review the output manually in PyMOL or a text editor.")

# -------------------------------------------------------------------
# Best-effort cleanup of common auxiliary files
# -------------------------------------------------------------------
for ext in [".log", ".propka", ".in"]:
    temp_file = output_dir / input_receptor.name.replace(".pdb", ext)
    if temp_file.exists():
        try:
            temp_file.unlink()
        except Exception:
            pass

# -------------------------------------------------------------------
# Report
# -------------------------------------------------------------------
print("\nReceptor chemical state defined successfully:")
print(f"  Input receptor:       {input_receptor.name}")
print(f"  Force field:          {forcefield}")
print(f"  Target pH:            {target_ph}")
print(f"  Protonated output:    {receptor_pqr_path.name}")
print(f"  Atom records written: {atom_count}")
print(f"  H-like atom names:    {hydrogen_like_count}")

# Optional: update working variable for downstream steps
receptor_pqr = receptor_pqr_path

Using cleaned receptor:
  File: PPARG_P37231_231_505.B99990002_renum_uniprot_cleaned.pdb
  Path: C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\homology\PPARG_P37231_231_505.B99990002_renum_uniprot_cleaned.pdb

Running PDB2PQR...
Command:
C:\Users\Usuario\.conda\envs\bio_env\Scripts\pdb2pqr30.EXE --ff AMBER --with-ph 7.4 C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\homology\PPARG_P37231_231_505.B99990002_renum_uniprot_cleaned.pdb C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\homology\PPARG_P37231_231_505.B99990002_renum_uniprot_cleaned_pH74.pqr

Receptor chemical state defined successfully:
  Input receptor:       PPARG_P37231_231_505.B99990002_renum_uniprot_cleaned.pdb
  Force field:          AMBER
  Target pH:            7.4
  Protonated output:    PPARG_P37231_231_505.B99990002_renum_uniprot_cleaned_pH74.pqr
  Atom records written: 4486
  H-like atom names:    2276


> **Output note:**  
> In this step, the cleaned receptor structure is converted into a chemically defined form by adding hydrogens and assigning protonation states under near-physiological conditions. This confirms that the receptor can be processed successfully with **PDB2PQR** and that the selected preparation settings are compatible with the final reconstructed model.  
>
> In this example, the cleaned receptor `PPARG_P37231_231_505.B99990002_renum_uniprot_cleaned.pdb` was processed using the **AMBER** force field at **pH 7.4**. The procedure completed successfully and generated the protonated receptor file `PPARG_P37231_231_505.B99990002_renum_uniprot_cleaned_pH74.pqr`.  
>
> The resulting prepared structure contained **4486 atom records**, including **2276 hydrogen-like atom names**, indicating that hydrogen addition and protonation were effectively applied during receptor preparation.  
>
> In practical terms, this step provides a receptor with an explicitly defined chemical state, suitable for subsequent ligand compatibility checks and for the next stages of docking-file preparation.

## Step 4A. Inspect and hydrogenate the reference ligand

After defining the chemical state of the receptor, the next step is to prepare the crystallographic ligand that will be used as a structural reference for docking. At this stage, the ligand is loaded and inspected to confirm that its geometry, connectivity, and 3D coordinates are chemically reasonable.

Explicit hydrogens are then added to the ligand so that its structure can be evaluated in a chemically complete form before further refinement. The hydrogenated ligand should be inspected visually to confirm that atom connectivity and hydrogen placement are consistent with the expected molecular structure.

In practical terms, this step generates an intermediate ligand structure that is chemically more complete and ready for geometry refinement in the following minimization step.

In [11]:
# Step 4A. Inspect and hydrogenate the reference ligand
# Use Open Babel to add hydrogens at near-physiological pH and generate
# an intermediate ligand PDB for visual inspection before minimization.

from pathlib import Path
import shutil
import subprocess

# -------------------------------------------------------------------
# Auto-detect project root
# -------------------------------------------------------------------
notebook_dir = Path.cwd().resolve()
project_root = None

for parent in [notebook_dir] + list(notebook_dir.parents):
    if (parent / "docking" / "prepared").exists():
        project_root = parent
        break

if project_root is None:
    raise FileNotFoundError(
        "\nCould not locate project root.\n"
        "   Make sure you're running this notebook from within the AutodockTutorial folder."
    )

# -------------------------------------------------------------------
# Locate ligand file
# -------------------------------------------------------------------
prepared_dir = project_root / "docking" / "prepared"
ligand_input_path = prepared_dir / "5YCP_BRL_ligand.pdb"

if not ligand_input_path.exists():
    raise FileNotFoundError(
        f"\nLigand file not found:\n   {ligand_input_path}\n\n"
        "   Expected crystallographic ligand file: 5YCP_BRL_ligand.pdb"
    )

print("Using ligand:")
print(f"  File: {ligand_input_path.name}")
print(f"  Path: {ligand_input_path.resolve()}")

# -------------------------------------------------------------------
# Locate Open Babel executable
# -------------------------------------------------------------------
obabel_exe = shutil.which("obabel") or shutil.which("babel")

if obabel_exe is None:
    raise EnvironmentError(
        "\nOpen Babel was not found in the active environment.\n"
        "   Please install Open Babel and make sure 'obabel' is available from the command line."
    )

# -------------------------------------------------------------------
# Define output file
# -------------------------------------------------------------------
target_ph = 7.4
ligand_h_path = prepared_dir / "5YCP_BRL_ligand_withH.pdb"

# -------------------------------------------------------------------
# Hydrogenate ligand with Open Babel
# -------------------------------------------------------------------
cmd = [
    obabel_exe,
    str(ligand_input_path),
    "-O", str(ligand_h_path),
    "-p", str(target_ph),
]

print("\nRunning Open Babel...")
print("Command:")
print(" ".join(cmd))

try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=300
    )
except subprocess.TimeoutExpired:
    raise TimeoutError(
        "\nOpen Babel timed out after 300 seconds.\n"
        "   Check the ligand file or environment configuration."
    )

if result.returncode != 0:
    raise RuntimeError(
        "\nOpen Babel failed during ligand hydrogenation.\n\n"
        f"STDOUT:\n{result.stdout}\n\n"
        f"STDERR:\n{result.stderr}"
    )

if not ligand_h_path.exists():
    raise FileNotFoundError(
        f"\nHydrogenated ligand file was not created:\n   {ligand_h_path}"
    )

# -------------------------------------------------------------------
# Basic validation of output PDB
# -------------------------------------------------------------------
atom_count = 0
hetatm_count = 0
hydrogen_like_count = 0

with open(ligand_h_path, "r", encoding="utf-8") as fh:
    for line in fh:
        if line.startswith("ATOM"):
            atom_count += 1
            atom_name = line[12:16].strip()
            element = line[76:78].strip()
            if atom_name.startswith("H") or element == "H":
                hydrogen_like_count += 1

        elif line.startswith("HETATM"):
            hetatm_count += 1
            atom_name = line[12:16].strip()
            element = line[76:78].strip()
            if atom_name.startswith("H") or element == "H":
                hydrogen_like_count += 1

total_coordinate_records = atom_count + hetatm_count

if total_coordinate_records == 0:
    raise ValueError(
        "\nThe hydrogenated ligand PDB contains no ATOM/HETATM records.\n"
        "   Inspect the Open Babel output file."
    )

if hydrogen_like_count == 0:
    print("\nWarning: No hydrogen atoms were detected in the output PDB.")
    print("   Open Babel may not have added hydrogens as expected.")
    print("   Inspect the file manually before proceeding.")

# -------------------------------------------------------------------
# Report
# -------------------------------------------------------------------
print("\nLigand hydrogenation summary")
print("-" * 60)
print(f"Input ligand:             {ligand_input_path.name}")
print(f"Hydrogenated output:      {ligand_h_path.name}")
print(f"Target pH:                {target_ph}")
print(f"ATOM records:             {atom_count}")
print(f"HETATM records:           {hetatm_count}")
print(f"Total coordinate records: {total_coordinate_records}")
print(f"Hydrogen-like atoms:      {hydrogen_like_count}")

print("\nVisual inspection is recommended before minimization.")
print("   Open the output PDB in PyMOL and confirm that:")
print("   - the ligand connectivity is reasonable")
print("   - hydrogens were added correctly")
print("   - no abnormal geometry was introduced")

# Optional: update working variable for downstream steps
ligand_hydrogenated_path = ligand_h_path

Using ligand:
  File: 5YCP_BRL_ligand.pdb
  Path: C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\prepared\5YCP_BRL_ligand.pdb

Running Open Babel...
Command:
C:\Program Files\OpenBabel-2.4.1\obabel.EXE C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\prepared\5YCP_BRL_ligand.pdb -O C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\prepared\5YCP_BRL_ligand_withH.pdb -p 7.4

Ligand hydrogenation summary
------------------------------------------------------------
Input ligand:             5YCP_BRL_ligand.pdb
Hydrogenated output:      5YCP_BRL_ligand_withH.pdb
Target pH:                7.4
ATOM records:             19
HETATM records:           25
Total coordinate records: 44
Hydrogen-like atoms:      19

Visual inspection is recommended before minimization.
   Open the output PDB in PyMOL and confirm that:
   - the ligand connectivity is reasonable
   - hydrogens were added correctly
   - no abnormal geometry was introduced


> **Output note:**  
> In this step, the crystallographic ligand is hydrogenated under near-physiological conditions to generate an intermediate structure suitable for visual inspection before energy minimization. This confirms that the ligand can be processed successfully with **Open Babel** and that a hydrogenated working file can be produced for the next stage of preparation.  
>
> In this example, the input ligand `5YCP_BRL_ligand.pdb` was processed at **pH 7.4**, generating the hydrogenated output file `5YCP_BRL_ligand_withH.pdb`. The resulting structure contained **44 coordinate records** in total, including **19 hydrogen-like atoms**, indicating that hydrogen addition was successfully applied.  
>
> At this stage, the hydrogenated ligand should be inspected visually in **PyMOL** to confirm that the connectivity remains reasonable, hydrogens were added in plausible positions, and no abnormal geometry was introduced during preparation.  
>
> In practical terms, this step yields a hydrogenated ligand structure ready for visual validation and subsequent geometry minimization prior to docking-file conversion.

## Step 4B. Minimize the ligand geometry

Once the hydrogenated ligand has been inspected and no obvious connectivity or geometry problems are observed, the next step is to refine its structure by energy minimization. This helps relieve unfavorable contacts introduced during hydrogen addition and produces a more relaxed ligand conformation for downstream docking preparation.

In this workflow, minimization is carried out with **Open Babel** using the `obminimize` utility and the **MMFF94** force field. The purpose of this step is not to generate a completely new ligand conformation, but to obtain a geometrically improved version of the hydrogenated structure while preserving its overall chemical identity.

The output of this step is a **minimized ligand PDB file** that can be inspected again in **PyMOL** before conversion to the docking format required by the selected engine. For typical organic ligands, **MMFF94** is a reasonable choice, although ligands containing metals or unsupported atom types may require a different force field or preparation workflow.

In [13]:
# Step 4B. Minimize the ligand geometry
# Use Open Babel's obminimize to relax the hydrogenated ligand geometry
# with the MMFF94 force field before docking-file conversion.

from pathlib import Path
import shutil
import subprocess

# -------------------------------------------------------------------
# Auto-detect project root
# -------------------------------------------------------------------
notebook_dir = Path.cwd().resolve()
project_root = None

for parent in [notebook_dir] + list(notebook_dir.parents):
    if (parent / "docking" / "prepared").exists():
        project_root = parent
        break

if project_root is None:
    raise FileNotFoundError(
        "\nCould not locate project root.\n"
        "   Make sure you're running this notebook from within the AutodockTutorial folder."
    )

prepared_dir = project_root / "docking" / "prepared"

# -------------------------------------------------------------------
# Resolve input ligand from Step 4A or from disk
# -------------------------------------------------------------------
if "ligand_hydrogenated_path" in globals():
    ligand_input_path = Path(ligand_hydrogenated_path)
else:
    ligand_input_path = prepared_dir / "5YCP_BRL_ligand_withH.pdb"

if not ligand_input_path.exists():
    raise FileNotFoundError(
        f"\nHydrogenated ligand file not found:\n   {ligand_input_path}\n\n"
        "   Please run Step 4A first, or make sure '5YCP_BRL_ligand_withH.pdb' exists."
    )

print("Using hydrogenated ligand:")
print(f"  File: {ligand_input_path.name}")
print(f"  Path: {ligand_input_path.resolve()}")

# -------------------------------------------------------------------
# Locate obminimize executable
# -------------------------------------------------------------------
obminimize_exe = shutil.which("obminimize")

if obminimize_exe is None:
    raise EnvironmentError(
        "\nobminimize was not found in the active environment.\n"
        "   Please install Open Babel and make sure 'obminimize' is available from the command line."
    )

# -------------------------------------------------------------------
# Define output file and minimization settings
# -------------------------------------------------------------------
ligand_min_path = prepared_dir / "5YCP_BRL_ligand_min.pdb"
forcefield = "MMFF94"
max_steps = 20000
convergence = "1e-6"

cmd = [
    obminimize_exe,
    "-ff", forcefield,
    "-n", str(max_steps),
    "-c", convergence,
    str(ligand_input_path),
]

print("\nRunning obminimize...")
print("Command:")
print(" ".join(cmd))
print(f"Output file: {ligand_min_path.name}")

# -------------------------------------------------------------------
# Run minimization and write stdout to output PDB
# -------------------------------------------------------------------
try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=600
    )
except subprocess.TimeoutExpired:
    raise TimeoutError(
        "\nobminimize timed out after 600 seconds.\n"
        "   Check the ligand file or the Open Babel installation."
    )

if result.returncode != 0:
    raise RuntimeError(
        "\nobminimize failed.\n\n"
        f"STDOUT:\n{result.stdout}\n\n"
        f"STDERR:\n{result.stderr}"
    )

with open(ligand_min_path, "w", encoding="utf-8") as fh:
    fh.write(result.stdout)

if not ligand_min_path.exists():
    raise FileNotFoundError(
        f"\nMinimized ligand file was not created:\n   {ligand_min_path}"
    )

# -------------------------------------------------------------------
# Basic validation of minimized output
# -------------------------------------------------------------------
atom_count = 0
hetatm_count = 0
hydrogen_like_count = 0

with open(ligand_min_path, "r", encoding="utf-8") as fh:
    for line in fh:
        if line.startswith("ATOM"):
            atom_count += 1
            atom_name = line[12:16].strip()
            element = line[76:78].strip()
            if atom_name.startswith("H") or element == "H":
                hydrogen_like_count += 1

        elif line.startswith("HETATM"):
            hetatm_count += 1
            atom_name = line[12:16].strip()
            element = line[76:78].strip()
            if atom_name.startswith("H") or element == "H":
                hydrogen_like_count += 1

total_coordinate_records = atom_count + hetatm_count

if total_coordinate_records == 0:
    raise ValueError(
        "\nThe minimized ligand file contains no ATOM/HETATM records.\n"
        "   Inspect the obminimize output manually."
    )

# -------------------------------------------------------------------
# Report
# -------------------------------------------------------------------
print("\nLigand minimization summary")
print("-" * 60)
print(f"Input ligand:             {ligand_input_path.name}")
print(f"Minimized output:         {ligand_min_path.name}")
print(f"Force field:              {forcefield}")
print(f"Maximum steps:            {max_steps}")
print(f"Convergence criterion:    {convergence}")
print(f"ATOM records:             {atom_count}")
print(f"HETATM records:           {hetatm_count}")
print(f"Total coordinate records: {total_coordinate_records}")
print(f"Hydrogen-like atoms:      {hydrogen_like_count}")

print("\nNote: MMFF94 is appropriate for typical organic ligands.")
print("   Ligands containing metals or unsupported elements may require")
print("   a different force field or a different preparation workflow.")

print("\nVisual inspection is recommended after minimization.")
print("   Compare the minimized ligand against the hydrogenated input in PyMOL")
print("   to confirm that the geometry was relaxed without introducing distortions.")

# Optional: update working variable for downstream steps
ligand_minimized_path = ligand_min_path

Using hydrogenated ligand:
  File: 5YCP_BRL_ligand_withH.pdb
  Path: C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\prepared\5YCP_BRL_ligand_withH.pdb

Running obminimize...
Command:
C:\Program Files\OpenBabel-2.4.1\obminimize.EXE -ff MMFF94 -n 20000 -c 1e-6 C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\prepared\5YCP_BRL_ligand_withH.pdb
Output file: 5YCP_BRL_ligand_min.pdb

Ligand minimization summary
------------------------------------------------------------
Input ligand:             5YCP_BRL_ligand_withH.pdb
Minimized output:         5YCP_BRL_ligand_min.pdb
Force field:              MMFF94
Maximum steps:            20000
Convergence criterion:    1e-6
ATOM records:             19
HETATM records:           25
Total coordinate records: 44
Hydrogen-like atoms:      19

Note: MMFF94 is appropriate for typical organic ligands.
   Ligands containing metals or unsupported elements may require
   a different force field or a different preparation workf

> **Output note:**  
> In this step, the hydrogenated ligand is refined by energy minimization in order to relax its geometry before docking-file preparation. This confirms that the ligand can be processed successfully with **Open Babel** using `obminimize` and that the selected force field is compatible with the current ligand structure.  
>
> In this example, the hydrogenated input file `5YCP_BRL_ligand_withH.pdb` was minimized with the **MMFF94** force field using a maximum of **20,000 steps** and a convergence criterion of **1e-6**. The procedure completed successfully and generated the minimized ligand file `5YCP_BRL_ligand_min.pdb`.  
>
> The resulting minimized structure retained **44 coordinate records** in total, including **19 hydrogen-like atoms**, indicating that the ligand was minimized without loss of atoms during the process.  
>
> For typical organic ligands, **MMFF94** is a reasonable force field for this stage, although ligands containing metals or unsupported atom types may require a different preparation strategy.  
>
> In practical terms, this step yields a hydrogenated and geometrically relaxed ligand structure that can now be inspected visually and carried forward to docking-format conversion.

## Step 5A. Convert the receptor to docking format

In this step, the protonated receptor (in **PQR format**) generated in Step 3 is converted into the file format required by the docking engine. For AutoDock Vina–based workflows, this means generating a **PDBQT** file that contains atom types and partial charges compatible with the AutoDock scoring function.

Because the receptor was saved as a **PQR** file (which includes charges and radii in additional columns), an intermediate **PDB** file is first reconstructed while preserving the atomic coordinates and restoring the standard element column required by downstream preparation tools. **The charges stored in the PQR file are intentionally discarded at this stage**, since the receptor-preparation tool will assign the charge model required by the selected docking workflow.

This intermediate PDB is then passed to a receptor-preparation tool such as **Meeko** (`mk_prepare_receptor.py`) or **ADFR** (`prepare_receptor`) to generate the final **PDBQT** file with AutoDock-compatible atom types.

At this stage, the goal is not to modify the receptor fold, but to produce a docking-compatible representation. A basic validation confirms that the output file was created successfully and contains the expected atomic records.

In practical terms, this step converts the chemically prepared receptor from **PQR** into its final docking-ready **PDBQT** representation.

In [ ]:
# Step 5A. Convert the receptor to docking format
# Convert the protonated receptor from PQR to a hydrogenated PDB,
# then generate the receptor PDBQT for docking using external atom rules.

from pathlib import Path
import shutil
import subprocess
import yaml

# -------------------------------------------------------------------
# Auto-detect project root
# -------------------------------------------------------------------
notebook_dir = Path.cwd().resolve()
project_root = None

for parent in [notebook_dir] + list(notebook_dir.parents):
    if (parent / "docking" / "homology").exists():
        project_root = parent
        break

if project_root is None:
    raise FileNotFoundError(
        "\nCould not locate project root.\n"
        "   Make sure you're running this notebook from within the AutodockTutorial folder."
    )

homology_dir = project_root / "docking" / "homology"
config_dir = project_root / "docking" / "config"
atom_rules_path = config_dir / "atom_rules.yaml"

# -------------------------------------------------------------------
# Load atom rules from YAML
# -------------------------------------------------------------------
if not atom_rules_path.exists():
    raise FileNotFoundError(
        f"\nAtom rules file not found:\n   {atom_rules_path}\n\n"
        "   Expected file: docking/config/atom_rules.yaml"
    )

with open(atom_rules_path, "r", encoding="utf-8") as fh:
    atom_rules = yaml.safe_load(fh)

if not isinstance(atom_rules, dict):
    raise ValueError(
        "\natom_rules.yaml could not be parsed as a valid dictionary."
    )

if "protein_atom_names" not in atom_rules or "two_letter_elements" not in atom_rules:
    raise KeyError(
        "\natom_rules.yaml must contain the keys:\n"
        "   - protein_atom_names\n"
        "   - two_letter_elements"
    )

PROTEIN_ATOM_PREFIXES = set(atom_rules["protein_atom_names"])
TWO_LETTER_ELEMENTS = {
    str(k).upper(): str(v)
    for k, v in atom_rules["two_letter_elements"].items()
}

print("Atom rules loaded successfully:")
print(f"  File: {atom_rules_path.name}")
print(f"  Protein atom names: {len(PROTEIN_ATOM_PREFIXES)}")
print(f"  Two-letter elements: {len(TWO_LETTER_ELEMENTS)}")

# -------------------------------------------------------------------
# Resolve protonated receptor from Step 3 or from disk
# -------------------------------------------------------------------
if "receptor_pqr_path" in globals():
    input_receptor_pqr = Path(receptor_pqr_path)
else:
    candidates = sorted(homology_dir.glob("*_cleaned_pH*.pqr"))
    if not candidates:
        raise FileNotFoundError(
            f"\nNo protonated receptor PQR file was found in:\n   {homology_dir}\n\n"
            "   Please run Step 3 first."
        )
    input_receptor_pqr = candidates[-1]

if not input_receptor_pqr.exists():
    raise FileNotFoundError(
        f"\nProtonated receptor file not found:\n   {input_receptor_pqr}"
    )

print("\nUsing protonated receptor:")
print(f"  File: {input_receptor_pqr.name}")
print(f"  Path: {input_receptor_pqr.resolve()}")

# -------------------------------------------------------------------
# Define intermediate and output files
# IMPORTANT:
# Avoid Path.with_suffix() on multi-dot names for final output creation.
# -------------------------------------------------------------------
receptor_pdb_path = input_receptor_pqr.parent / f"{input_receptor_pqr.stem}_for_docking.pdb"
receptor_pdbqt_path = input_receptor_pqr.parent / f"{input_receptor_pqr.stem}.pdbqt"

# -------------------------------------------------------------------
# Infer element from atom name using external rules
# -------------------------------------------------------------------
def infer_element_from_atom_name(atom_name: str) -> str:
    atom = atom_name.strip().upper()

    if not atom:
        return ""

    if atom in PROTEIN_ATOM_PREFIXES:
        return atom[0]

    for prefix, element in TWO_LETTER_ELEMENTS.items():
        if atom.startswith(prefix):
            return element

    return atom[0]

# -------------------------------------------------------------------
# Convert PQR to PDB
# Keep coordinates from the PQR and discard charge/radius columns.
# -------------------------------------------------------------------
atom_records = 0

with open(input_receptor_pqr, "r", encoding="utf-8") as fin, open(
    receptor_pdb_path, "w", encoding="utf-8"
) as fout:
    for line in fin:
        if line.startswith(("ATOM", "HETATM")):
            atom_records += 1
            core = line[:66].rstrip("\n")
            atom_name = line[12:16]
            element = line[76:78].strip() if len(line) >= 78 else ""

            if not element:
                element = infer_element_from_atom_name(atom_name)

            pdb_line = f"{core:<76}{element:>2}\n"
            fout.write(pdb_line)

        elif line.startswith(("TER", "END")):
            fout.write(line if line.endswith("\n") else line + "\n")

if atom_records == 0:
    raise ValueError(
        "\nThe input PQR file does not contain ATOM/HETATM records.\n"
        "   Inspect the protonated receptor before proceeding."
    )

if not receptor_pdb_path.exists():
    raise FileNotFoundError(
        f"\nIntermediate receptor PDB file was not created:\n   {receptor_pdb_path}"
    )

print("\nIntermediate hydrogenated PDB created:")
print(f"  File: {receptor_pdb_path.name}")
print(f"  Path: {receptor_pdb_path.resolve()}")

print("\nNote:")
print("  The AMBER charges stored in the PQR file are not preserved in the PDB file.")
print("  This is expected in this workflow, because docking preparation tools")
print("  will assign the charge model required for PDBQT generation.")

# -------------------------------------------------------------------
# Detect receptor-preparation tool
# Priority:
# 1) mk_prepare_receptor.py
# 2) mk_prepare_receptor
# 3) prepare_receptor
# -------------------------------------------------------------------
prepare_tool = (
    shutil.which("mk_prepare_receptor.py")
    or shutil.which("mk_prepare_receptor")
    or shutil.which("prepare_receptor")
)

if prepare_tool is None:
    raise EnvironmentError(
        "\nNo receptor-preparation tool was found.\n"
        "   Install either Meeko (mk_prepare_receptor.py / mk_prepare_receptor)\n"
        "   or ADFR Suite (prepare_receptor), and make sure it is available in PATH."
    )

print("\nDetected preparation tool:")
print(f"  {prepare_tool}")

# -------------------------------------------------------------------
# Build command according to the detected tool
# -------------------------------------------------------------------
tool_name = Path(prepare_tool).name.lower()

if "mk_prepare_receptor" in tool_name:
    output_base = receptor_pdbqt_path.parent / input_receptor_pqr.stem
    cmd = [
        prepare_tool,
        "-i", str(receptor_pdb_path),
        "-o", str(output_base),
        "-p",
    ]
    expected_pdbqt = output_base.parent / f"{output_base.name}.pdbqt"
else:
    cmd = [
        prepare_tool,
        "-r", str(receptor_pdb_path),
        "-o", str(receptor_pdbqt_path),
    ]
    expected_pdbqt = receptor_pdbqt_path

print("\nRunning receptor preparation...")
print("Command:")
print(" ".join(cmd))

try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=600
    )
except subprocess.TimeoutExpired:
    raise TimeoutError(
        "\nReceptor preparation timed out after 600 seconds.\n"
        "   Check the receptor file or the external tool installation."
    )

if result.returncode != 0:
    raise RuntimeError(
        "\nReceptor preparation failed.\n\n"
        f"STDOUT:\n{result.stdout}\n\n"
        f"STDERR:\n{result.stderr}"
    )

print("\nPreparation tool finished.")
if result.stdout.strip():
    print("\nSTDOUT:")
    print(result.stdout)
if result.stderr.strip():
    print("\nSTDERR:")
    print(result.stderr)

if not expected_pdbqt.exists():
    # Extra diagnostic: list nearby pdbqt files in the output folder
    nearby_pdbqt = sorted(expected_pdbqt.parent.glob("*.pdbqt"))
    diagnostic = "\n".join(str(p.name) for p in nearby_pdbqt) if nearby_pdbqt else "None found"
    raise FileNotFoundError(
        "\nReceptor PDBQT file was not created at the expected path.\n"
        f"   Expected:\n   {expected_pdbqt}\n\n"
        f"   PDBQT files currently found in {expected_pdbqt.parent}:\n   {diagnostic}"
    )

# -------------------------------------------------------------------
# Basic validation of output PDBQT
# -------------------------------------------------------------------
atom_count = 0

with open(expected_pdbqt, "r", encoding="utf-8") as fh:
    for line in fh:
        if line.startswith(("ATOM", "HETATM")):
            atom_count += 1

if atom_count == 0:
    raise ValueError(
        "\nThe receptor PDBQT file contains no ATOM/HETATM records.\n"
        "   Inspect the output before proceeding."
    )

# -------------------------------------------------------------------
# Report
# -------------------------------------------------------------------
print("\nReceptor docking-format conversion summary")
print("-" * 60)
print(f"Input receptor (PQR):     {input_receptor_pqr.name}")
print(f"Intermediate PDB:         {receptor_pdb_path.name}")
print(f"Preparation tool:         {Path(prepare_tool).name}")
print(f"Output receptor PDBQT:    {expected_pdbqt.name}")
print(f"Full path:                {expected_pdbqt.resolve()}")
print(f"ATOM/HETATM records:      {atom_count}")

# Optional: update working variables for downstream steps
receptor_docking_pdb_path = receptor_pdb_path
receptor_pdbqt_path = expected_pdbqt

Atom rules loaded successfully:
  File: atom_rules.yaml
  Protein atom names: 45
  Two-letter elements: 9

Using protonated receptor:
  File: PPARG_P37231_231_505.B99990002_renum_uniprot_cleaned_pH74.pqr
  Path: C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\homology\PPARG_P37231_231_505.B99990002_renum_uniprot_cleaned_pH74.pqr

Intermediate hydrogenated PDB created:
  File: PPARG_P37231_231_505.B99990002_renum_uniprot_cleaned_pH74_for_docking.pdb
  Path: C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\homology\PPARG_P37231_231_505.B99990002_renum_uniprot_cleaned_pH74_for_docking.pdb

Note:
  The AMBER charges stored in the PQR file are not preserved in the PDB file.
  This is expected in this workflow, because docking preparation tools
  will assign the charge model required for PDBQT generation.

Detected preparation tool:
  C:\Users\Usuario\.conda\envs\bio_env\Scripts\mk_prepare_receptor.EXE

Running receptor preparation...
Command:
C:\Users\Usuari

> **Output note:**  
> In this step, the protonated receptor generated in **Step 3** (in **PQR** format) was successfully converted into the docking-ready format required for AutoDock Vina. The workflow first reconstructed an intermediate **PDB** file from the **PQR** receptor while preserving the atomic coordinates and restoring the element column required by downstream preparation tools. As expected for this workflow, the charges stored in the **PQR** file were not carried forward into the intermediate **PDB** representation.
>
> The receptor was then processed with **Meeko** (`mk_prepare_receptor`), which generated the final **PDBQT** file in the `docking/homology/` directory: `PPARG_P37231_231_505.B99990002_renum_uniprot_cleaned_pH74.pdbqt`. The resulting file contained **2684 ATOM/HETATM records**, confirming that the receptor was written successfully in a complete docking-compatible representation.
>
> In practical terms, this step confirms that the chemically prepared receptor is now available in its final **PDBQT** format and can be used in the subsequent docking setup steps.

## Step 5B. Convert the reference ligand to docking format

In this step, the minimized **reference ligand** is converted into the docking format required by the selected engine. For AutoDock Vina workflows, this means generating a **PDBQT** file from the ligand structure prepared in **Step 4B**.

The starting point is the minimized ligand in **PDB** format. This structure is passed to **Open Babel** to generate a **PDBQT** file while preserving the ligand identity, retaining all hydrogens, and assigning AutoDock-compatible atom types during the conversion.

At this stage, the objective is not to alter the ligand geometry further, but to produce a docking-compatible ligand representation that can be used together with the prepared receptor. A basic validation is also performed to confirm that the output file was created successfully and contains the expected atomic records and torsion tree information.

In practical terms, this step generates the final **PDBQT** file for the crystallographic or reference ligand, which can be used for docking setup, protocol validation, or comparison with external test ligands prepared later in the workflow.

In [4]:
# Step 5B. Convert the ligand to docking format
# Convert the minimized ligand PDB into PDBQT for AutoDock Vina,
# preserving hydrogens and atom identity information as much as possible.

from pathlib import Path
import shutil
import subprocess

# -------------------------------------------------------------------
# Auto-detect project root
# -------------------------------------------------------------------
notebook_dir = Path.cwd().resolve()
project_root = None

for parent in [notebook_dir] + list(notebook_dir.parents):
    if (parent / "docking" / "prepared").exists():
        project_root = parent
        break

if project_root is None:
    raise FileNotFoundError(
        "\n Could not locate project root.\n"
        "   Make sure you're running this notebook from within the AutodockTutorial folder."
    )

prepared_dir = project_root / "docking" / "prepared"

# -------------------------------------------------------------------
# Resolve minimized ligand from Step 4B or from disk
# -------------------------------------------------------------------
if "ligand_min_path" in globals():
    input_ligand = Path(ligand_min_path)
else:
    candidates = sorted(prepared_dir.glob("*_min.pdb"))
    if not candidates:
        raise FileNotFoundError(
            f"\n No minimized ligand file was found in:\n   {prepared_dir}\n\n"
            "   Please run Step 4B first."
        )
    input_ligand = candidates[-1]

if not input_ligand.exists():
    raise FileNotFoundError(
        f"\n Minimized ligand file not found:\n   {input_ligand}"
    )

print("Using minimized ligand:")
print(f"  File: {input_ligand.name}")
print(f"  Path: {input_ligand.resolve()}")

# -------------------------------------------------------------------
# Detect Open Babel
# -------------------------------------------------------------------
obabel_exe = shutil.which("obabel") or shutil.which("babel")

if obabel_exe is None:
    raise EnvironmentError(
        "\n Open Babel was not found in the active environment.\n"
        "   Please install Open Babel and make sure 'obabel' or 'babel'\n"
        "   is available from the command line."
    )

# -------------------------------------------------------------------
# Define output file
# -------------------------------------------------------------------
ligand_pdbqt_path = input_ligand.parent / f"{input_ligand.stem}.pdbqt"

if ligand_pdbqt_path.exists():
    ligand_pdbqt_path.unlink()

# -------------------------------------------------------------------
# Build Open Babel command
# -xh : preserve hydrogens
# -xn : preserve atom names
# -xp : preserve atom indices
# -------------------------------------------------------------------
cmd = [
    obabel_exe,
    str(input_ligand),
    "-O", str(ligand_pdbqt_path),
    "-xh",
    "-xn",
    "-xp",
]

print("\nRunning Open Babel...")
print("Command:")
print(" ".join(cmd))

try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=300
    )
except subprocess.TimeoutExpired:
    raise TimeoutError(
        "\n Open Babel timed out after 300 seconds.\n"
        "   Check the ligand file or the Open Babel installation."
    )

if result.returncode != 0:
    raise RuntimeError(
        "\n Ligand conversion to PDBQT failed.\n\n"
        f"STDOUT:\n{result.stdout}\n\n"
        f"STDERR:\n{result.stderr}"
    )

if not ligand_pdbqt_path.exists():
    raise FileNotFoundError(
        f"\n Ligand PDBQT file was not created:\n   {ligand_pdbqt_path}"
    )

# -------------------------------------------------------------------
# Count atoms in input PDB
# -------------------------------------------------------------------
input_atom_count = 0
with open(input_ligand, "r", encoding="utf-8") as fh:
    for line in fh:
        if line.startswith(("ATOM", "HETATM")):
            input_atom_count += 1

if input_atom_count == 0:
    raise ValueError(
        "\n The minimized ligand PDB contains no ATOM/HETATM records.\n"
        "   Inspect the input ligand before proceeding."
    )

# -------------------------------------------------------------------
# Basic validation of output PDBQT
# -------------------------------------------------------------------
output_atom_count = 0
tree_records = 0
remark_lines = 0

with open(ligand_pdbqt_path, "r", encoding="utf-8") as fh:
    for line in fh:
        if line.startswith(("ATOM", "HETATM")):
            output_atom_count += 1
        elif line.startswith(("ROOT", "BRANCH", "ENDBRANCH", "TORSDOF")):
            tree_records += 1
        elif line.startswith("REMARK"):
            remark_lines += 1

if output_atom_count == 0:
    raise ValueError(
        "\n The ligand PDBQT file contains no ATOM/HETATM records.\n"
        "   Inspect the output before proceeding."
    )

# -------------------------------------------------------------------
# Report
# -------------------------------------------------------------------
print("\nLigand docking-format conversion summary")
print("-" * 60)
print(f"Input ligand:             {input_ligand.name}")
print(f"Output ligand PDBQT:      {ligand_pdbqt_path.name}")
print(f"Full path:                {ligand_pdbqt_path.resolve()}")
print(f"Input ATOM/HETATM:        {input_atom_count}")
print(f"Output ATOM/HETATM:       {output_atom_count}")
print(f"Tree-related records:     {tree_records}")
print(f"REMARK lines:             {remark_lines}")

if output_atom_count != input_atom_count:
    print("\n Warning: Atom count changed during conversion.")
    print(f"   Input atoms:  {input_atom_count}")
    print(f"   Output atoms: {output_atom_count}")
    print("   Inspect the ligand carefully before docking.")
else:
    print("\n Atom count was preserved during conversion.")

print("\n Note:")
print("   This PDBQT conversion step is intended for AutoDock Vina workflows.")
print("   If you plan to use AutoDock4, ligand charge handling may require")
print("   a different preparation route.")

print("\n Visual inspection is recommended.")
print("   Open the ligand PDBQT in PyMOL or a text editor and confirm that:")
print("   - atom records are present")
print("   - hydrogens were preserved as expected")
print("   - the file is not empty or truncated")
print("   - the ligand identity is preserved after conversion")

# Optional: update working variable for downstream steps
ligand_pdbqt_path = ligand_pdbqt_path

Using minimized ligand:
  File: 5YCP_BRL_ligand_min.pdb
  Path: C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\prepared\5YCP_BRL_ligand_min.pdb

Running Open Babel...
Command:
C:\Program Files\OpenBabel-2.4.1\obabel.EXE C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\prepared\5YCP_BRL_ligand_min.pdb -O C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\prepared\5YCP_BRL_ligand_min.pdbqt -xh -xn -xp

Ligand docking-format conversion summary
------------------------------------------------------------
Input ligand:             5YCP_BRL_ligand_min.pdb
Output ligand PDBQT:      5YCP_BRL_ligand_min.pdbqt
Full path:                C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\prepared\5YCP_BRL_ligand_min.pdbqt
Input ATOM/HETATM:        44
Output ATOM/HETATM:       44
Tree-related records:     18
REMARK lines:             13

 Atom count was preserved during conversion.

 Note:
   This PDBQT conversion step is intended for Au

> **Output note:**  
> In this step, the minimized reference ligand was successfully converted from **PDB** to **PDBQT**, generating the docking-ready ligand file required for AutoDock Vina. The conversion was performed with **Open Babel**, which automatically preserves hydrogens, atom names, and atom indices during PDBQT generation.  
>
> In this example, the input file `5YCP_BRL_ligand_min.pdb` was converted into `5YCP_BRL_ligand_min.pdbqt` and saved in the `docking/prepared/` directory. The resulting **PDBQT** file contained **44 ATOM/HETATM records**, which is the expected atom count for the fully hydrogenated BRL ligand (rosiglitazone, C₁₈H₁₉N₃O₃S). This indicates that atom preservation during conversion was successful and that the ligand identity was maintained consistently in the docking-ready file.  
>
> Additional records related to the ligand tree representation (`ROOT`, `BRANCH`, `ENDBRANCH`, `TORSDOF`) were also written, which is expected in the **PDBQT** format for ligand flexibility handling. In practical terms, this step confirms that the reference ligand is now available in its final docking-compatible representation and can be used for docking setup, protocol validation, or comparison with external ligands prepared later in the workflow.

## Step 5C. Prepare an External Ligand from SDF for Docking

For ligands other than the co-crystallized ligand present in the receptor structure, the preparation workflow accepts an external **SDF** file containing three-dimensional coordinates. Such files are commonly available from chemical databases such as **PubChem** and **ZINC**.

The preparation proceeds in three sequential steps:

1. **Step 5C1 – Protonation (SDF → PDB)**  
   The SDF file is converted to **PDB** format using **Open Babel**. Hydrogens are added and protonation states are assigned at **pH 7.4** using the `-p 7.4` option. The output is a PDB file containing the ligand with explicit hydrogens and a protonation state consistent with physiologically reasonable conditions.

2. **Step 5C2 – Energy Minimization (PDB → Minimized PDB)**  
   The protonated PDB structure is subjected to energy minimization using the **MMFF94** force field as implemented in Open Babel’s `obminimize` utility. A maximum of **20,000 steps** is specified together with a convergence criterion of **1 × 10⁻⁶**. This step relaxes bond lengths, angles, and torsions toward a local energy minimum before docking.

3. **Step 5C3 – Docking Format Conversion (Minimized PDB → PDBQT)**  
   The minimized PDB structure is converted to **PDBQT** format using **Open Babel**. In this format, non-polar hydrogens bonded to carbon are typically merged into the parent heavy atom, whereas polar hydrogens bonded to heteroatoms are retained explicitly. The output also includes the atom-type assignments and torsion-tree definitions required by **AutoDock Vina**.

Validation of the final **PDBQT** file includes verification of atom-record count, presence of torsion-tree records, and confirmation that the total partial charge is chemically reasonable for the expected ligand charge state.

Together, these three steps produce a protonated, minimized, and docking-ready ligand structure in **PDBQT** format suitable for molecular docking with **AutoDock Vina**.

### Input Requirements

| Requirement | Description |
|-------------|-------------|
| **File format** | SDF (Structure Data File) |
| **File location** | `docking/sdf/` |
| **3D coordinates** | Required (Open Babel does not generate 3D from 2D) |
| **Protonation state** | Not required in input (Step 5C1 adds hydrogens at pH 7.4) |


### How to Specify the Input Ligand

You can provide the input ligand in one of two ways:

1. **Automatic detection** (recommended for a single file):  
   Place exactly one `.sdf` file in the `docking/sdf/` directory. The notebook will automatically detect and use it.

2. **Explicit specification** (required for multiple files):  
   If multiple `.sdf` files are present, define the variable before running this step:
   
   ```python
   user_ligand_sdf = "CID_10198397.sdf"
   ```

In [17]:
# Step 5C1. Convert an external ligand from SDF to PDB with hydrogens

from pathlib import Path
import shutil
import subprocess

# -------------------------------------------------------------------
# User-configurable variable
# Example:
# user_ligand_sdf = "CID_10198397.sdf"
# -------------------------------------------------------------------
user_ligand_sdf = None

# -------------------------------------------------------------------
# Auto-detect project root
# -------------------------------------------------------------------
notebook_dir = Path.cwd().resolve()
project_root = None

for parent in [notebook_dir] + list(notebook_dir.parents):
    if (parent / "docking" / "sdf").exists():
        project_root = parent
        break

if project_root is None:
    raise FileNotFoundError(
        "\n Could not locate project root.\n"
        "   Make sure you're running this notebook from within the AutodockTutorial folder."
    )

sdf_dir = project_root / "docking" / "sdf"

if not sdf_dir.exists():
    raise FileNotFoundError(
        f"\n Required folder not found:\n   {sdf_dir}\n\n"
        "   Create the folder 'docking/sdf/' and place your ligand .sdf file there."
    )

# -------------------------------------------------------------------
# Select input SDF
# -------------------------------------------------------------------
if user_ligand_sdf is not None:
    user_ligand_sdf_path = Path(user_ligand_sdf)
    input_sdf = user_ligand_sdf_path if user_ligand_sdf_path.is_absolute() else sdf_dir / user_ligand_sdf_path.name
else:
    sdf_candidates = sorted(sdf_dir.glob("*.sdf"))

    if len(sdf_candidates) == 0:
        raise FileNotFoundError(
            f"\n No SDF ligand file was found in:\n   {sdf_dir}\n\n"
            "   Place your ligand .sdf file there, or set:\n"
            "   user_ligand_sdf = 'your_file.sdf'"
        )
    elif len(sdf_candidates) > 1:
        raise ValueError(
            "\n Multiple SDF files were found in docking/sdf/.\n"
            "   Please set the variable:\n"
            "   user_ligand_sdf = 'your_file.sdf'\n\n"
            f"   Detected files:\n   " + "\n   ".join(p.name for p in sdf_candidates)
        )
    else:
        input_sdf = sdf_candidates[0]

if not input_sdf.exists():
    raise FileNotFoundError(f"\n Input SDF file not found:\n   {input_sdf}")

if input_sdf.suffix.lower() != ".sdf":
    raise ValueError(f"\n Expected an SDF file, but received:\n   {input_sdf.name}")

print("Using external ligand:")
print(f"  File: {input_sdf.name}")
print(f"  Path: {input_sdf.resolve()}")

# -------------------------------------------------------------------
# Detect Open Babel
# -------------------------------------------------------------------
obabel_exe = shutil.which("obabel") or shutil.which("babel")

if obabel_exe is None:
    raise EnvironmentError(
        "\n Open Babel was not found in the active environment.\n"
        "   Please install Open Babel and make sure 'obabel' or 'babel' is available."
    )

# -------------------------------------------------------------------
# Define output PDB
# -------------------------------------------------------------------
ligand_pdb_path = input_sdf.with_suffix(".pdb")

if ligand_pdb_path.exists():
    ligand_pdb_path.unlink()

# -------------------------------------------------------------------
# Build command: SDF -> PDB with hydrogens
# -------------------------------------------------------------------
cmd = [
    obabel_exe,
    str(input_sdf),
    "-O", str(ligand_pdb_path),
    "-p", "7.4",
]

print("\nRunning Open Babel...")
print("Command:")
print(" ".join(cmd))

try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=300
    )
except subprocess.TimeoutExpired:
    raise TimeoutError(
        "\n SDF-to-PDB conversion timed out after 300 seconds.\n"
        "   Check the ligand file or the Open Babel installation."
    )

if result.returncode != 0:
    raise RuntimeError(
        "\n SDF-to-PDB conversion failed.\n\n"
        f"STDOUT:\n{result.stdout}\n\n"
        f"STDERR:\n{result.stderr}"
    )

if not ligand_pdb_path.exists():
    raise FileNotFoundError(
        f"\n Output PDB file was not created:\n   {ligand_pdb_path}"
    )

# -------------------------------------------------------------------
# Basic validation of output PDB
# -------------------------------------------------------------------
atom_count = 0
hydrogen_count = 0

with open(ligand_pdb_path, "r", encoding="utf-8") as fh:
    for line in fh:
        if line.startswith(("ATOM", "HETATM")):
            atom_count += 1
            atom_name = line[12:16].strip()
            element = line[76:78].strip()
            if atom_name.startswith("H") or element in ("H", "D"):
                hydrogen_count += 1

if atom_count == 0:
    raise ValueError(
        "\n The generated PDB contains no ATOM/HETATM records.\n"
        "   Inspect the output before proceeding."
    )

print("\nExternal ligand conversion summary")
print("-" * 60)
print(f"Input SDF ligand:          {input_sdf.name}")
print(f"Output PDB ligand:         {ligand_pdb_path.name}")
print(f"Full path:                 {ligand_pdb_path.resolve()}")
print(f"ATOM/HETATM records:       {atom_count}")
print(f"Hydrogen-like atoms:       {hydrogen_count}")

print("\n Visual inspection is recommended before minimization.")
print("   Open the output PDB in PyMOL and confirm that:")
print("   - the ligand connectivity is reasonable")
print("   - hydrogens are present")
print("   - no abnormal geometry was introduced")

print("\n Note:")
print("   This workflow assumes that the input ligand is a 3D SDF file.")
print("   If the SDF lacks reliable 3D coordinates, inspect the structure")
print("   carefully before minimization and docking.")


# Optional variable for downstream steps
external_ligand_pdb_path = ligand_pdb_path

Using external ligand:
  File: CID_10198397.sdf
  Path: C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\sdf\CID_10198397.sdf

Running Open Babel...
Command:
C:\Program Files\OpenBabel-2.4.1\obabel.EXE C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\sdf\CID_10198397.sdf -O C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\sdf\CID_10198397.pdb -p 7.4

External ligand conversion summary
------------------------------------------------------------
Input SDF ligand:          CID_10198397.sdf
Output PDB ligand:         CID_10198397.pdb
Full path:                 C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\sdf\CID_10198397.pdb
ATOM/HETATM records:       24
Hydrogen-like atoms:       9

 Visual inspection is recommended before minimization.
   Open the output PDB in PyMOL and confirm that:
   - the ligand connectivity is reasonable
   - hydrogens are present
   - no abnormal geometry was introduced

 Note:
   This workflow a

In [18]:
# Step 5C2. Minimize the external ligand PDB

from pathlib import Path
import shutil
import subprocess

# -------------------------------------------------------------------
# Input from Step 5C1
# -------------------------------------------------------------------
if "external_ligand_pdb_path" not in globals():
    raise NameError(
        "Variable 'external_ligand_pdb_path' was not found. Please run Step 5C1 first."
    )

input_ligand = Path(external_ligand_pdb_path)

if not input_ligand.exists():
    raise FileNotFoundError(
        f"\n Input ligand PDB file not found:\n   {input_ligand}"
    )

print("Using external ligand PDB:")
print(f"  File: {input_ligand.name}")
print(f"  Path: {input_ligand.resolve()}")

# -------------------------------------------------------------------
# Detect obminimize
# -------------------------------------------------------------------
obminimize_exe = shutil.which("obminimize")

if obminimize_exe is None:
    raise EnvironmentError(
        "\n obminimize was not found in the active environment.\n"
        "   Please install Open Babel and make sure 'obminimize' is available."
    )

# -------------------------------------------------------------------
# Output file
# -------------------------------------------------------------------
ligand_min_path = input_ligand.with_name(f"{input_ligand.stem}_min.pdb")

if ligand_min_path.exists():
    ligand_min_path.unlink()

# -------------------------------------------------------------------
# Minimization settings
# -------------------------------------------------------------------
force_field = "MMFF94"
max_steps = 20000
convergence = "1e-6"

cmd = [
    obminimize_exe,
    "-ff", force_field,
    "-n", str(max_steps),
    "-c", convergence,
    str(input_ligand),
]

print("\nRunning obminimize...")
print("Command:")
print(" ".join(cmd))
print(f"Output file: {ligand_min_path.name}")

try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=600
    )
except subprocess.TimeoutExpired:
    raise TimeoutError(
        "\n Ligand minimization timed out after 600 seconds.\n"
        "   Check the ligand structure or the Open Babel installation."
    )

if result.returncode != 0:
    raise RuntimeError(
        "\n Ligand minimization failed.\n\n"
        f"STDOUT:\n{result.stdout}\n\n"
        f"STDERR:\n{result.stderr}"
    )

# obminimize writes minimized structure to STDOUT
if not result.stdout.strip():
    raise ValueError(
        "\n obminimize did not return a minimized structure.\n"
        "   Inspect the input ligand before proceeding."
    )

with open(ligand_min_path, "w", encoding="utf-8") as fh:
    fh.write(result.stdout)

if not ligand_min_path.exists():
    raise FileNotFoundError(
        f"\n Minimized ligand file was not created:\n   {ligand_min_path}"
    )

# -------------------------------------------------------------------
# Basic validation
# -------------------------------------------------------------------
atom_count = 0
hydrogen_count = 0

with open(ligand_min_path, "r", encoding="utf-8") as fh:
    for line in fh:
        if line.startswith(("ATOM", "HETATM")):
            atom_count += 1
            atom_name = line[12:16].strip()
            element = line[76:78].strip()
            if atom_name.startswith("H") or element in ("H", "D"):
                hydrogen_count += 1

if atom_count == 0:
    raise ValueError(
        "\n The minimized ligand contains no ATOM/HETATM records.\n"
        "   Inspect the output before proceeding."
    )

print("\nLigand minimization summary")
print("-" * 60)
print(f"Input ligand:             {input_ligand.name}")
print(f"Minimized output:         {ligand_min_path.name}")
print(f"Force field:              {force_field}")
print(f"Maximum steps:            {max_steps}")
print(f"Convergence criterion:    {convergence}")
print(f"ATOM/HETATM records:      {atom_count}")
print(f"Hydrogen-like atoms:      {hydrogen_count}")

print("\n Note: MMFF94 is appropriate for typical organic ligands.")
print("   Ligands containing metals or unsupported elements may require")
print("   a different force field or a different preparation workflow.")

print("\n Visual inspection is recommended after minimization.")
print("   Compare the minimized ligand against the initial PDB in PyMOL")
print("   to confirm that the geometry was relaxed without distortions.")

# Optional variable for downstream steps
external_ligand_min_path = ligand_min_path

Using external ligand PDB:
  File: CID_10198397.pdb
  Path: C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\sdf\CID_10198397.pdb

Running obminimize...
Command:
C:\Program Files\OpenBabel-2.4.1\obminimize.EXE -ff MMFF94 -n 20000 -c 1e-6 C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\sdf\CID_10198397.pdb
Output file: CID_10198397_min.pdb

Ligand minimization summary
------------------------------------------------------------
Input ligand:             CID_10198397.pdb
Minimized output:         CID_10198397_min.pdb
Force field:              MMFF94
Maximum steps:            20000
Convergence criterion:    1e-6
ATOM/HETATM records:      24
Hydrogen-like atoms:      9

 Note: MMFF94 is appropriate for typical organic ligands.
   Ligands containing metals or unsupported elements may require
   a different force field or a different preparation workflow.

 Visual inspection is recommended after minimization.
   Compare the minimized ligand against the initia

In [19]:
# Step 5C3. Convert the minimized external ligand to PDBQT

from pathlib import Path
import shutil
import subprocess

# -------------------------------------------------------------------
# Input from Step 5C2
# -------------------------------------------------------------------
if "external_ligand_min_path" not in globals():
    raise NameError(
        "Variable 'external_ligand_min_path' was not found. Please run Step 5C2 first."
    )

input_ligand = Path(external_ligand_min_path)

if not input_ligand.exists():
    raise FileNotFoundError(
        f"\n Minimized ligand file not found:\n   {input_ligand}"
    )

print("Using minimized external ligand:")
print(f"  File: {input_ligand.name}")
print(f"  Path: {input_ligand.resolve()}")

# -------------------------------------------------------------------
# Detect Open Babel
# -------------------------------------------------------------------
obabel_exe = shutil.which("obabel") or shutil.which("babel")

if obabel_exe is None:
    raise EnvironmentError(
        "\n Open Babel was not found in the active environment.\n"
        "   Please install Open Babel and make sure 'obabel' or 'babel' is available."
    )

# -------------------------------------------------------------------
# Output file
# -------------------------------------------------------------------
ligand_pdbqt_path = input_ligand.with_suffix(".pdbqt")

if ligand_pdbqt_path.exists():
    ligand_pdbqt_path.unlink()



# -------------------------------------------------------------------
# Build command: PDB -> PDBQT
# Sin flags adicionales para permitir que Babel haga su trabajo por defecto:
# 1. Calcule cargas Gasteiger (necesarias para Vina)
# 2. Preserve todos los hidrógenos (All-atom)
# -------------------------------------------------------------------
cmd = [
    obabel_exe,
    "-ipdb", str(input_ligand),
    "-opdbqt",
    "-O", str(ligand_pdbqt_path),
    # Eliminamos -xn, -xp y -xh para no interferir con el cálculo de cargas
    # y la preservación de hidrógenos polares.
]

print("\nRunning Open Babel...")
print("Command:")
print(" ".join(cmd))

try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=300
    )
except subprocess.TimeoutExpired:
    raise TimeoutError(
        "\n Ligand PDBQT conversion timed out after 300 seconds.\n"
        "   Check the ligand file or the Open Babel installation."
    )

if result.returncode != 0:
    raise RuntimeError(
        "\n Ligand PDBQT conversion failed.\n\n"
        f"STDOUT:\n{result.stdout}\n\n"
        f"STDERR:\n{result.stderr}"
    )

if not ligand_pdbqt_path.exists():
    raise FileNotFoundError(
        f"\n Ligand PDBQT file was not created:\n   {ligand_pdbqt_path}"
    )

# -------------------------------------------------------------------
# Validation: compare atom counts
# -------------------------------------------------------------------
input_atoms = 0
with open(input_ligand, "r", encoding="utf-8") as fh:
    for line in fh:
        if line.startswith(("ATOM", "HETATM")):
            input_atoms += 1

output_atoms = 0
tree_records = 0
remark_lines = 0

with open(ligand_pdbqt_path, "r", encoding="utf-8") as fh:
    for line in fh:
        if line.startswith(("ATOM", "HETATM")):
            output_atoms += 1
        elif line.startswith(("ROOT", "BRANCH", "ENDBRANCH", "TORSDOF")):
            tree_records += 1
        elif line.startswith("REMARK"):
            remark_lines += 1

if output_atoms == 0:
    raise ValueError(
        "\n The generated ligand PDBQT contains no ATOM/HETATM records.\n"
        "   Inspect the output before proceeding."
    )

print("\nExternal ligand docking-format conversion summary")
print("-" * 60)
print(f"Input ligand:             {input_ligand.name}")
print(f"Output ligand PDBQT:      {ligand_pdbqt_path.name}")
print(f"Full path:                {ligand_pdbqt_path.resolve()}")
print(f"Input ATOM/HETATM:        {input_atoms}")
print(f"Output ATOM/HETATM:       {output_atoms}")
print(f"Tree-related records:     {tree_records}")
print(f"REMARK lines:             {remark_lines}")

if input_atoms == output_atoms:
    print("\n Atom count was preserved during conversion.")
else:
    print("\n Note: Atom count changed during conversion.")
    print(f"   Input atoms:  {input_atoms}")
    print(f"   Output atoms: {output_atoms}")
    print("   This is EXPECTED for AutoDock PDBQT format:")
    print("   Non-polar hydrogens (C-H) are represented implicitly")
    print("   by the atom types, not as explicit atoms.")
    print("   However, inspect the ligand carefully before docking.")

print("\n Visual inspection is recommended.")
print("   Open the ligand PDBQT in PyMOL or a text editor and confirm that:")
print("   - atom records are present")
print("   - the file is not empty or truncated")
print("   - the ligand identity is preserved after conversion")
print("   - the ligand is ready for docking with the prepared receptor")

# Optional variable for downstream steps
external_ligand_pdbqt_path = ligand_pdbqt_path

Using minimized external ligand:
  File: CID_10198397_min.pdb
  Path: C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\sdf\CID_10198397_min.pdb

Running Open Babel...
Command:
C:\Program Files\OpenBabel-2.4.1\obabel.EXE -ipdb C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\sdf\CID_10198397_min.pdb -opdbqt -O C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\sdf\CID_10198397_min.pdbqt

External ligand docking-format conversion summary
------------------------------------------------------------
Input ligand:             CID_10198397_min.pdb
Output ligand PDBQT:      CID_10198397_min.pdbqt
Full path:                C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\sdf\CID_10198397_min.pdbqt
Input ATOM/HETATM:        24
Output ATOM/HETATM:       17
Tree-related records:     8
REMARK lines:             8

 Note: Atom count changed during conversion.
   Input atoms:  24
   Output atoms: 17
   This is EXPECTED for AutoDock PDBQT 

In [ ]:
# Revise the PDBQT file (Open Babel step)
from pathlib import Path

pdbqt_path = Path(
    r"C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\sdf\CID_10198397_min.pdbqt")

atom_lines = []
hydrogen_lines = []
total_charge = 0.0

def extract_charge_from_pdbqt_line(line: str):
    """
    Extract partial charge from PDBQT.
    According to the PDBQT column definition:
    partialChrg = columns 71-76 -> Python slice [70:76]
    """
    if len(line) >= 76:
        charge_str = line[70:76].strip()
        try:
            return float(charge_str)
        except ValueError:
            pass

    # Fallback: whitespace-based parsing
    fields = line.split()
    if len(fields) >= 2:
        possible_charge = fields[-2]
        try:
            return float(possible_charge)
        except ValueError:
            pass

    return None

with open(pdbqt_path, "r", encoding="utf-8") as fh:
    for line in fh:
        if line.startswith(("ATOM", "HETATM")):
            clean_line = line.rstrip()
            atom_lines.append(clean_line)

            atom_name = line[12:16].strip()

            # PDBQT atom type: columns 79-80 -> Python slice [78:80]
            atom_type = line[78:80].strip() if len(line) >= 80 else ""

            # Fallback if alignment is irregular
            if not atom_type:
                parts = line.split()
                if parts:
                    atom_type = parts[-1].strip()

            if atom_name.startswith("H") or atom_type in ("H", "HD", "D"):
                hydrogen_lines.append(clean_line)

            charge = extract_charge_from_pdbqt_line(line)
            if charge is not None:
                total_charge += charge

print(f"Total ATOM/HETATM lines: {len(atom_lines)}")
print(f"Explicit hydrogen atoms:    {len(hydrogen_lines)}")
print(f"TOTAL Charge calculated = {total_charge:.4f}")

print("\nFirst ATOM/HETATM records:")
for line in atom_lines[:20]:
    print(line)

if hydrogen_lines:
    print("\nExplicit hydrogen records (polar hydrogens only):")
    for line in hydrogen_lines:
        print(line)
else:
    print("\nNo explicit hydrogen records found (non-polar hydrogens are implicit).")

Total ATOM/HETATM lines: 17
Explicit hydrogen atoms:    2
TOTAL Charge calculated = -0.0010

First ATOM/HETATM records:
ATOM      1  C   UNL     1       0.279  -0.682   0.837  0.00  0.00    +0.059 C
ATOM      2  S   UNL     1       1.708   1.344  -0.481  0.00  0.00    -0.054 S
ATOM      3  C   UNL     1       3.412   1.043  -0.272  0.00  0.00    +0.279 C
ATOM      4  N   UNL     1       3.641  -0.279  -0.108  0.00  0.00    -0.246 N
ATOM      5  H   UNL     1       4.577  -0.611   0.066  0.00  0.00    +0.158 HD
ATOM      6  C   UNL     1       2.578  -1.126  -0.040  0.00  0.00    +0.234 C
ATOM      7  O   UNL     1       2.696  -2.320   0.206  0.00  0.00    -0.274 OA
ATOM      8  O   UNL     1       4.261   1.917  -0.254  0.00  0.00    -0.263 OA
ATOM      9  C   UNL     1       1.263  -0.422  -0.305  0.00  0.00    +0.146 C
ATOM     10  C   UNL     1      -1.148  -0.331   0.483  0.00  0.00    -0.046 A
ATOM     11  C   UNL     1      -1.959  -1.266  -0.176  0.00  0.00    +0.007 A
ATOM    

In [ ]:
# Step 5D. Prepare an external ligand from SDF using Meeko
# Alternative workflow to compare against the Open Babel route.

from pathlib import Path
import shutil
import subprocess

# -------------------------------------------------------------------
# User-configurable variable
# If multiple SDF files exist in docking/sdf/, specify one explicitly.
# Example:
# user_ligand_sdf = "CID_10198397.sdf"
# -------------------------------------------------------------------
user_ligand_sdf = None

# -------------------------------------------------------------------
# Auto-detect project root
# -------------------------------------------------------------------
notebook_dir = Path.cwd().resolve()
project_root = None

for parent in [notebook_dir] + list(notebook_dir.parents):
    if (parent / "docking" / "sdf").exists():
        project_root = parent
        break

if project_root is None:
    raise FileNotFoundError(
        "\n Could not locate project root.\n"
        "   Make sure you are running this notebook from within the AutodockTutorial folder."
    )

sdf_dir = project_root / "docking" / "sdf"

if not sdf_dir.exists():
    raise FileNotFoundError(
        f"\n Required folder not found:\n   {sdf_dir}\n\n"
        "   Create the folder 'docking/sdf/' and place your ligand .sdf file there."
    )

# -------------------------------------------------------------------
# Select input SDF
# -------------------------------------------------------------------
if user_ligand_sdf is not None:
    user_ligand_sdf_path = Path(user_ligand_sdf)
    input_sdf = user_ligand_sdf_path if user_ligand_sdf_path.is_absolute() else sdf_dir / user_ligand_sdf_path.name
else:
    sdf_candidates = sorted(sdf_dir.glob("*.sdf"))

    if len(sdf_candidates) == 0:
        raise FileNotFoundError(
            f"\n No SDF ligand file was found in:\n   {sdf_dir}\n\n"
            "   Place your ligand .sdf file there, or set:\n"
            "   user_ligand_sdf = 'your_file.sdf'"
        )
    elif len(sdf_candidates) > 1:
        raise ValueError(
            "\n Multiple SDF files were found in docking/sdf/.\n"
            "   Please set the variable:\n"
            "   user_ligand_sdf = 'your_file.sdf'\n\n"
            f"   Detected files:\n   " + "\n   ".join(p.name for p in sdf_candidates)
        )
    else:
        input_sdf = sdf_candidates[0]

if not input_sdf.exists():
    raise FileNotFoundError(f"\nInput SDF file not found:\n   {input_sdf}")

if input_sdf.suffix.lower() != ".sdf":
    raise ValueError(f"\nExpected an SDF file, but received:\n   {input_sdf.name}")

# -------------------------------------------------------------------
# Important note about input chemistry
# -------------------------------------------------------------------
print("Using external ligand:")
print(f"  File: {input_sdf.name}")
print(f"  Path: {input_sdf.resolve()}")

print("\nNote:")
print("  This Meeko workflow assumes the input SDF already contains a valid 3D structure")
print("  and a chemically reasonable protonation state.")
print("  If needed, protonation and energy minimization should be handled before this step.")

# -------------------------------------------------------------------
# Detect Meeko tool
# -------------------------------------------------------------------
prepare_tool = (
    shutil.which("mk_prepare_ligand")
    or shutil.which("mk_prepare_ligand.py")
)

if prepare_tool is None:
    raise EnvironmentError(
        "\n Meeko ligand-preparation tool was not found.\n"
        "   Please install Meeko in the active environment.\n\n"
        "   Example:\n"
        "   pip install meeko\n\n"
        "   Then verify with:\n"
        "   where mk_prepare_ligand"
    )

# -------------------------------------------------------------------
# Define output
# -------------------------------------------------------------------
ligand_pdbqt_path = input_sdf.with_name(input_sdf.stem + "_meeko.pdbqt")

if ligand_pdbqt_path.exists():
    ligand_pdbqt_path.unlink()

# -------------------------------------------------------------------
# Build command
# Keep the command minimal and safe.
# Do NOT pass bare -p, because in Meeko that can be interpreted as
# load_atom_params / pdbqt-related option depending on the entry point.
# -------------------------------------------------------------------
cmd = [
    prepare_tool,
    "-i", str(input_sdf),
    "-o", str(ligand_pdbqt_path),
]

print("\nDetected preparation tool:")
print(f"  {prepare_tool}")

print("\nRunning Meeko ligand preparation...")
print("Command:")
print(" ".join(cmd))

try:
    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        timeout=300
    )
except subprocess.TimeoutExpired:
    raise TimeoutError(
        "\n Ligand preparation timed out after 300 seconds.\n"
        "   Check the ligand file or the Meeko installation."
    )

if result.returncode != 0:
    raise RuntimeError(
        "\n Ligand preparation from SDF failed.\n\n"
        f"STDOUT:\n{result.stdout}\n\n"
        f"STDERR:\n{result.stderr}"
    )

if not ligand_pdbqt_path.exists():
    raise FileNotFoundError(
        f"\n Ligand PDBQT file was not created:\n   {ligand_pdbqt_path}"
    )

# -------------------------------------------------------------------
# Basic validation of output PDBQT
# PDBQT columns:
# partial charge ~ columns 71-76
# atom type ~ columns 79-80
# -------------------------------------------------------------------
atom_lines = []
hydrogen_lines = []
tree_records = 0
remark_lines = 0
total_charge = 0.0
charge_parse_errors = 0

with open(ligand_pdbqt_path, "r", encoding="utf-8") as fh:
    for line in fh:
        if line.startswith(("ATOM", "HETATM")):
            atom_lines.append(line.rstrip())

            atom_name = line[12:16].strip()
            atom_type = line[77:79].strip() if len(line) >= 79 else ""

            if atom_name.startswith("H") or atom_type in ("H", "HD", "HS"):
                hydrogen_lines.append(line.rstrip())

            try:
                partial_charge = float(line[70:76].strip())
                total_charge += partial_charge
            except ValueError:
                charge_parse_errors += 1

        elif line.startswith(("ROOT", "BRANCH", "ENDBRANCH", "TORSDOF")):
            tree_records += 1
        elif line.startswith("REMARK"):
            remark_lines += 1

atom_count = len(atom_lines)
hydrogen_count = len(hydrogen_lines)

if atom_count == 0:
    raise ValueError(
        "\n The generated ligand PDBQT contains no ATOM/HETATM records.\n"
        "   Inspect the output before proceeding."
    )

# -------------------------------------------------------------------
# Report
# -------------------------------------------------------------------
print("\nMeeko external ligand preparation summary")
print("-" * 60)
print(f"Input SDF ligand:          {input_sdf.name}")
print(f"Preparation tool:          {Path(prepare_tool).name}")
print(f"Output ligand PDBQT:       {ligand_pdbqt_path.name}")
print(f"Full path:                 {ligand_pdbqt_path.resolve()}")
print(f"ATOM/HETATM records:       {atom_count}")
print(f"Explicit hydrogens:        {hydrogen_count}")
print(f"Tree-related records:      {tree_records}")
print(f"REMARK lines:              {remark_lines}")
print(f"Total charge calculated:   {total_charge:.4f}")

if charge_parse_errors > 0:
    print(f"Charge parse warnings:     {charge_parse_errors}")

if tree_records == 0:
    print("\n  Note: No tree records were detected.")
    print("   The ligand may have no rotatable bonds, or the output should be inspected manually.")

print("\nVisual inspection is recommended.")
print("  Open the ligand PDBQT in PyMOL or a text editor and confirm that:")
print("  - atom records are present")
print("  - the file is not empty or truncated")
print("  - the ligand identity is preserved")
print("  - explicit polar hydrogens are reasonable")
print("  - the ligand is ready for docking with the prepared receptor")

# Optional variable for downstream steps
external_ligand_meeko_pdbqt_path = ligand_pdbqt_path

Using external ligand:
  File: CID_10198397.sdf
  Path: C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\sdf\CID_10198397.sdf

Note:
  This Meeko workflow assumes the input SDF already contains a valid 3D structure
  and a chemically reasonable protonation state.
  If needed, protonation and energy minimization should be handled before this step.

Detected preparation tool:
  C:\Users\Usuario\.conda\envs\bio_env\Scripts\mk_prepare_ligand.EXE

Running Meeko ligand preparation...
Command:
C:\Users\Usuario\.conda\envs\bio_env\Scripts\mk_prepare_ligand.EXE -i C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\sdf\CID_10198397.sdf -o C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\sdf\CID_10198397_meeko.pdbqt

Meeko external ligand preparation summary
------------------------------------------------------------
Input SDF ligand:          CID_10198397.sdf
Preparation tool:          mk_prepare_ligand.EXE
Output ligand PDBQT:       CID_10198397_me

In [33]:
# Revise the PDBQT file (Meeko step)
from pathlib import Path

pdbqt_path = Path(
    r"C:\Users\Usuario\Documents\w_TEC\GitHub\AutodockTutorial\docking\sdf\CID_10198397_meeko.pdbqt")

atom_lines = []
hydrogen_lines = []
total_charge = 0.0

def extract_charge_from_pdbqt_line(line: str):
    """
    Extract partial charge from PDBQT.
    According to the PDBQT column definition:
    partialChrg = columns 71-76 -> Python slice [70:76]
    """
    if len(line) >= 76:
        charge_str = line[70:76].strip()
        try:
            return float(charge_str)
        except ValueError:
            pass

    # Fallback: whitespace-based parsing
    fields = line.split()
    if len(fields) >= 2:
        possible_charge = fields[-2]
        try:
            return float(possible_charge)
        except ValueError:
            pass

    return None

with open(pdbqt_path, "r", encoding="utf-8") as fh:
    for line in fh:
        if line.startswith(("ATOM", "HETATM")):
            clean_line = line.rstrip()
            atom_lines.append(clean_line)

            atom_name = line[12:16].strip()

            # PDBQT atom type: columns 79-80 -> Python slice [78:80]
            atom_type = line[78:80].strip() if len(line) >= 80 else ""

            # Fallback if alignment is irregular
            if not atom_type:
                parts = line.split()
                if parts:
                    atom_type = parts[-1].strip()

            if atom_name.startswith("H") or atom_type in ("H", "HD", "D"):
                hydrogen_lines.append(clean_line)

            charge = extract_charge_from_pdbqt_line(line)
            if charge is not None:
                total_charge += charge

print(f"Total ATOM/HETATM lines: {len(atom_lines)}")
print(f"Explicit hydrogen atoms:    {len(hydrogen_lines)}")
print(f"TOTAL Charge calculated = {total_charge:.4f}")

print("\nFirst ATOM/HETATM records:")
for line in atom_lines[:20]:
    print(line)

if hydrogen_lines:
    print("\nExplicit hydrogen records (polar hydrogens only):")
    for line in hydrogen_lines:
        print(line)
else:
    print("\nNo explicit hydrogen records found (non-polar hydrogens are implicit).")

Total ATOM/HETATM lines: 17
Explicit hydrogen atoms:    2
TOTAL Charge calculated = -0.0010

First ATOM/HETATM records:
ATOM      1  C   UNL     1      -1.141  -0.340   0.474  1.00  0.00    -0.046 A
ATOM      2  C   UNL     1      -1.619   0.935   0.736  1.00  0.00     0.007 A
ATOM      3  C   UNL     1      -1.945  -1.298  -0.125  1.00  0.00     0.007 A
ATOM      4  C   UNL     1      -2.932   1.259   0.393  1.00  0.00     0.046 A
ATOM      5  C   UNL     1      -3.258  -0.974  -0.469  1.00  0.00     0.046 A
ATOM      6  C   UNL     1      -3.751   0.305  -0.210  1.00  0.00     0.115 A
ATOM      7  C   UNL     1       0.277  -0.690   0.845  1.00  0.00     0.060 C
ATOM      8  C   UNL     1       1.248  -0.397  -0.285  1.00  0.00     0.150 C
ATOM      9  S   UNL     1       1.694   1.367  -0.314  1.00  0.00    -0.051 SA
ATOM     10  C   UNL     1       3.412   1.063  -0.169  1.00  0.00     0.286 C
ATOM     11  C   UNL     1       2.548  -1.147  -0.103  1.00  0.00     0.241 C
ATOM     1